In [1]:
%load_ext autoreload
%autoreload 2

import os
os.chdir("C:/Users/Administrator/PythonProjects/abfluss_queich")

In [9]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import xarray as xr

from utils.logger import logger

from configs import settings
from jobs.icon.process_icon import clip_to_catchment, extract_precip_timeseries

icon_settings = settings.ingestion.icon
catchment_settings = settings.ingestion.catchment

In [131]:
input_dir = icon_settings.compressed_dir
input_dir = Path(input_dir)

catchment_path = catchment_settings.catchment_path
clip_crs = icon_settings.clip_crs

file_paths = sorted(p for p in input_dir.glob("*.grib2") if "_047" in p.stem)
# file_paths = sorted(p for p in input_dir.glob("*.grib2") if "_048" in p.stem)

catchment = gpd.read_file(catchment_path).to_crs(clip_crs)

In [132]:
def extract_precip_timeseries(
    precip: xr.DataArray,
) -> pd.DataFrame:

    spatial_dims = [
        d for d in precip.dims
        if d not in {"step", "time"}
    ]

    ts = precip.mean(
        dim=spatial_dims,
        skipna=True,
    )

    # Normalize scalar → 1D
    if ts.ndim == 0:
        ts = ts.expand_dims(
            valid_time=[pd.to_datetime(ts.valid_time.values)]
        )

    df = (
        ts
        .to_dataframe(name="precip_cum_mean")
        .reset_index()
        .set_index("valid_time")[["precip_cum_mean"]]
    )

    df.index = pd.to_datetime(df.index)
    df.index.name = "timestamp"

    return df

In [133]:
dfs: list[pd.DataFrame] = []

# --- Open dataset and extract ---
for file in file_paths[:1]:
    try:
        ds = xr.open_dataset(file, engine="cfgrib")

        precip_crop = clip_to_catchment(
            dataset=ds,
            catchment=catchment,
            crs=clip_crs
        )

        df_precip = extract_precip_timeseries(precip=precip_crop)

        dfs.append(df_precip)
    
    
    except Exception:
        logger.exception("Failed to process ICON GRIB file: %s", file)
        continue
    
    
    finally:
        ds.close()
    
        
    if not dfs:
        raise ValueError("No valid precipitation files processed.")
    